# Coffee Shop Sales — Data Cleaning & EDA
**Dataset:** Coffee Shop Sales — Kaggle  
**Period:** January 2023 – June 2023  
**Records:** 149,116 transactions  
**Stores:** BigBrew, Dunkin, Starbucks  
**Tools:** Python, Pandas, NumPy  

**Project Goal:**  
Clean and prepare raw coffee shop sales data for analysis  
and visualization in Power BI. This notebook covers data  
cleaning, feature engineering, validation, and EDA.

## 1.) Import Libraries
Importing the necessary libraries for data manipulation and analysis.

In [1]:
import pandas as pd
import numpy as np
import os 


## 2. Data Cleaning Pipeline
All cleaning steps are wrapped in a single reusable function.
This ensures the process is repeatable every time the raw data updates.

**Steps inside the pipeline:**
- Strip whitespace from all text columns
- Rename and map store IDs to store names
- Map store locations to Philippine regions
- Feature engineering (Month, Year, Hour, Day, Day_parts, Total_Amount)
- Apply categorical ordering for proper sorting
- Reset index

In [2]:
def clean_coffee_data(filepath):
    """
    Data cleaning pipeline for Coffee Shop Sales dataset.
    Source: Kaggle
    """
    
    # Load
    df = pd.read_excel(filepath)
    print(f"✅ Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
    
    # Strip whitespace from all text columns
    for col in df.columns:
        try:
            df[col] = df[col].str.strip()
        except AttributeError:
            pass
    
    # Rename store_id to Store_Name
    df.rename(columns={'store_id': 'Store_Name'}, inplace=True)
    
    # Map store names and locations
    df['Store_Name'] = df['Store_Name'].replace({
        8: 'BigBrew', 5: 'Dunkin', 3: 'Starbucks'
    })
    df['store_location'] = df['store_location'].replace({
        'Lower Manhattan': 'Luzon',
        "Hell's Kitchen": 'Visayas',
        'Astoria': 'Mindanao'
    })

    # Feature Engineering
    df['Month'] = df['transaction_date'].dt.month_name()
    df['Month_Number'] = df['transaction_date'].dt.month
    df['Year'] = df['transaction_date'].dt.year
    df['Total_Amount'] = df['transaction_qty'] * df['unit_price']
    df['Timestamp_Convert'] = pd.to_timedelta(df['transaction_time'].astype(str))
    df['Hour'] = df['Timestamp_Convert'].dt.seconds // 3600
    df['Day'] = df['transaction_date'].dt.day_name()

    # Day parts
    def assign_dayparts(hour):
        if 5 <= hour <= 7:
            return 'Early Morning'
        elif 8 <= hour <= 10:
            return 'Mid-Morning'
        elif 11 <= hour <= 13:
            return 'Lunch'
        elif 14 <= hour <= 17:
            return 'Afternoon'
        else:
            return 'Evening'

    df['Day_parts'] = df['Hour'].apply(assign_dayparts)

    # Categorical ordering
    df['Month'] = pd.Categorical(df['Month'], categories=[
        'January','February','March','April','May','June',
        'July','August','September','October','November','December'
    ], ordered=True)

    df['Day'] = pd.Categorical(df['Day'], categories=[
        'Monday','Tuesday','Wednesday','Thursday',
        'Friday','Saturday','Sunday'
    ], ordered=True)

    df['Day_parts'] = pd.Categorical(df['Day_parts'], categories=[
        'Early Morning','Mid-Morning','Lunch','Afternoon','Evening'
    ], ordered=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)

    # Final summary
    print(f"✅ Cleaned: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(f"📅 Date range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
    print(f"🏪 Stores: {df['Store_Name'].unique().tolist()}")
    print(f"💰 Total Revenue: {df['Total_Amount'].sum():,.2f}")

    return df

In [3]:
# Run the pipeline
df = clean_coffee_data('Coffee Shop Sales.xlsx')

✅ Loaded: 149,116 rows, 11 columns
✅ Cleaned: 149,116 rows, 19 columns
📅 Date range: 2023-01-01 to 2023-06-30
🏪 Stores: ['Dunkin', 'BigBrew', 'Starbucks']
💰 Total Revenue: 698,812.33


## 3.) Initial Exploration
Quick look at the cleaned data structure, shape, statistics and sample rows.


In [4]:
df.info()
# Basically checking the Datetypes and how many rows and columns we have.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype          
---  ------             --------------   -----          
 0   transaction_id     149116 non-null  int64          
 1   transaction_date   149116 non-null  datetime64[ns] 
 2   transaction_time   149116 non-null  object         
 3   transaction_qty    149116 non-null  int64          
 4   Store_Name         149116 non-null  object         
 5   store_location     149116 non-null  object         
 6   product_id         149116 non-null  int64          
 7   unit_price         149116 non-null  float64        
 8   product_category   149116 non-null  object         
 9   product_type       149116 non-null  object         
 10  product_detail     149116 non-null  object         
 11  Month              149116 non-null  category       
 12  Month_Number       149116 non-null  int32          
 13  Year               149116 non

In [5]:
df.describe()
# Trying to look for insights with our integer columns.

,transaction_id,transaction_date,transaction_qty,product_id,unit_price,Month_Number,Year,Total_Amount,Timestamp_Convert,Hour
count,149116.000000,149116,149116.000000,149116.000000,149116.000000,149116.000000,149116.0,149116.000000,149116,149116.000000
mean,74737.371872,2023-04-15 11:50:32.173609984,1.438276,47.918607,3.382219,3.988881,2023.0,4.686367,0 days 12:14:15.815794415,11.735790
min,1.000000,2023-01-01 00:00:00,1.000000,1.000000,0.800000,1.000000,2023.0,0.800000,0 days 06:00:00,6.000000
25%,37335.750000,2023-03-06 00:00:00,1.000000,33.000000,2.500000,3.000000,2023.0,3.000000,0 days 09:05:10.500000,9.000000
50%,74727.500000,2023-04-24 00:00:00,1.000000,47.000000,3.000000,4.000000,2023.0,3.750000,0 days 11:15:28,11.000000
75%,112094.250000,2023-05-30 00:00:00,2.000000,60.000000,3.750000,5.000000,2023.0,6.000000,0 days 15:25:57,15.000000
max,149456.000000,2023-06-30 00:00:00,8.000000,87.000000,45.000000,6.000000,2023.0,360.000000,0 days 20:59:32,20.000000
std,43153.600016,NaN,0.542509,17.930020,2.658723,1.673091,0.0,4.227099,0 days 03:45:57.901686173,3.764662


In [6]:
df.head()
# Mandatory check of how our Dataset looks like

,transaction_id,transaction_date,transaction_time,transaction_qty,Store_Name,store_location,product_id,unit_price,product_category,product_type,product_detail,Month,Month_Number,Year,Total_Amount,Timestamp_Convert,Hour,Day,Day_parts
0,1,2023-01-01,07:06:11,2,Dunkin,Luzon,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,January,1,2023,6.0,0 days 07:06:11,7,Sunday,Early Morning
1,2,2023-01-01,07:08:56,2,Dunkin,Luzon,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,January,1,2023,6.2,0 days 07:08:56,7,Sunday,Early Morning
2,3,2023-01-01,07:14:04,2,Dunkin,Luzon,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,January,1,2023,9.0,0 days 07:14:04,7,Sunday,Early Morning
3,4,2023-01-01,07:20:24,1,Dunkin,Luzon,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,January,1,2023,2.0,0 days 07:20:24,7,Sunday,Early Morning
4,5,2023-01-01,07:22:41,2,Dunkin,Luzon,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,January,1,2023,6.2,0 days 07:22:41,7,Sunday,Early Morning


## 4. Data Validation
Verifying data quality and integrity after cleaning.

### 4.1) Checking for duplicates to see if we have something to fix.

In [7]:
df.duplicated().sum()
# Luckily, there is no duplicate row.

np.int64(0)

### 4.2) Check for null values.

In [8]:
df.isnull().sum()
# There are no null values for any column as well. So we don't have anything to fix for this.

transaction_id       0
transaction_date     0
transaction_time     0
transaction_qty      0
Store_Name           0
store_location       0
product_id           0
unit_price           0
product_category     0
product_type         0
product_detail       0
Month                0
Month_Number         0
Year                 0
Total_Amount         0
Timestamp_Convert    0
Hour                 0
Day                  0
Day_parts            0
dtype: int64

In [9]:
df.nunique()
# Checking how many unique values I have for columns to have an idea what to look for later on.

transaction_id       149116
transaction_date        181
transaction_time      25762
transaction_qty           6
Store_Name                3
store_location            3
product_id               80
unit_price               41
product_category          9
product_type             29
product_detail           80
Month                     6
Month_Number              6
Year                      1
Total_Amount             75
Timestamp_Convert     25762
Hour                     15
Day                       7
Day_parts                 5
dtype: int64

In [10]:
df[['store_location','Store_Name']].drop_duplicates()
# I did check if there are other branches of the store in different locations. But, it shows here that there is only 1 store per location.

,store_location,Store_Name
0,Luzon,Dunkin
17,Visayas,BigBrew
105,Mindanao,Starbucks


In [11]:
df.transaction_id.duplicated().sum()
# I can confirm here that there are thousands of orders, but none of them have multiple products in one order.

np.int64(0)

In [12]:
# Set display max columns to 25 just in case we added more later, it doesn't collapse automatically. 
pd.set_option('display.max_columns' ,  25)

In [13]:
# We check if the same numbers timestamps were made.
df.transaction_time.nunique() - df.Timestamp_Convert.nunique()

0

In [14]:
# Check unique stores
df['Store_Name'].unique()

array(['Dunkin', 'BigBrew', 'Starbucks'], dtype=object)

In [15]:
# Check unique product categories
df['product_category'].unique()

array(['Coffee', 'Tea', 'Drinking Chocolate', 'Bakery', 'Flavours',
       'Loose Tea', 'Coffee beans', 'Packaged Chocolate', 'Branded'],
      dtype=object)

In [16]:
# Check unique months
df['Month'].unique()

['January', 'February', 'March', 'April', 'May', 'June']
Categories (12, object): ['January' < 'February' < 'March' < 'April' ... 'September' < 'October' < 'November' < 'December']

In [17]:
# Check unique day parts
df['Day_parts'].unique()

['Early Morning', 'Mid-Morning', 'Lunch', 'Afternoon', 'Evening']
Categories (5, object): ['Early Morning' < 'Mid-Morning' < 'Lunch' < 'Afternoon' < 'Evening']

In [18]:
# Check if transaction_ids are unique
df['transaction_id'].nunique() == len(df)

True

In [19]:
# Make sure transaction_qty is positive
print(df['transaction_qty'].min())
print(df['transaction_qty'].max())

# Make sure unit_price is positive
print(df['unit_price'].min())
print(df['unit_price'].max())

# Make sure Total_Amount is positive
print(df['Total_Amount'].min())
print(df['Total_Amount'].max())

1
8
0.8
45.0
0.8
360.0


In [20]:
# Confirm row and column count
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")
print(f"Stores: {df['Store_Name'].unique()}")
print(f"Total Revenue: {df['Total_Amount'].sum():,.2f}")

Rows: 149116
Columns: 19
Date range: 2023-01-01 00:00:00 to 2023-06-30 00:00:00
Stores: ['Dunkin' 'BigBrew' 'Starbucks']
Total Revenue: 698,812.33


## 5. Exploratory Data Analysis (EDA)
Analyzing patterns and trends in the cleaned data using
groupby aggregations and pivot tables.

### Total Sales per Store

In [21]:
agik = df.groupby('Store_Name',as_index=False).agg(Store_Total_Sales=('Total_Amount','sum'))
agik.sort_values('Store_Total_Sales', ascending=False)

,Store_Name,Store_Total_Sales
0,BigBrew,236511.17
2,Starbucks,232243.91
1,Dunkin,230057.25


### Getting the sales per prduct category

In [22]:
agik = df.groupby('product_category').agg(Total_Sales_Per_Category= ('Total_Amount','sum')
                                         ).sort_values('Total_Sales_Per_Category',ascending=False)

agik.Total_Sales_Per_Category.apply(lambda x:  f'{x:,.2f}').to_frame()

,Total_Sales_Per_Category
product_category,
Coffee,"269,952.45"
Tea,"196,405.95"
Bakery,"82,315.64"
Drinking Chocolate,"72,416.00"
Coffee beans,"40,085.25"
Branded,"13,607.00"
Loose Tea,"11,213.60"
Flavours,"8,408.80"
Packaged Chocolate,"4,407.64"


### Getting the sales per product category from different stores

In [23]:
# Using groupby to create a multi-level index. Primary Level would be the Store Name, and Secondary Level would be the product_category.
# Getting the total Sales per Category and sorting values from highest to lowest to make it easier to view which sells the most. 
# Using agg function to easily name the aggregated/derived column. After that, did another groupby to sort the value using apply + lambda. 

df.groupby(['Store_Name','product_category']).agg(Sales_Per_Category = ('Total_Amount' , 'sum')).groupby(level=0, group_keys=False).apply(
lambda x: x.sort_values('Sales_Per_Category', ascending=False)
)

Sales_Per_Category
Store_Name product_category                      
BigBrew    Coffee                        91222.65
           Tea                           64701.30
           Bakery                        27386.95
           Drinking Chocolate            23586.25
           Coffee beans                  18635.10
           Loose Tea                      4461.35
           Flavours                       2876.80
           Branded                        1942.00
           Packaged Chocolate             1698.77
Dunkin     Coffee                        88985.50
           Tea                           63864.75
           Bakery                        28328.94
           Drinking Chocolate            22494.50
           Coffee beans                  11230.95
           Branded                        6208.00
           Flavours                       3767.20
           Loose Tea                      3558.25
           Packaged Chocolate             1619.16
Starbucks  Coffee                        89744.30
           Tea                           67839.90
           Bakery                        26599.75
           Drinking Chocolate            26335.25
           Coffee beans                  10219.20
           Branded                        5457.00
           Loose Tea                      3194.00
           Flavours                       1764.80
           Packaged Chocolate             1089.71

In [24]:
# To be more specific with sales details, I go ahead with product detail this time. 
df.groupby(['product_category','product_detail']).agg(Sales_Per_Category = ('Total_Amount' , 'sum')).groupby(level=0, group_keys=False).apply(
lambda x: x.sort_values('Sales_Per_Category', ascending=False)
)

Sales_Per_Category
product_category product_detail                               
Bakery           Chocolate Croissant                  11625.98
                 Scottish Cream Scone                  8949.45
                 Ginger Scone                          8011.61
                 Jumbo Savory Scone                    7626.62
                 Almond Croissant                      7168.13
...                                                        ...
Tea              Traditional Blend Chai Rg            11280.00
                 Serenity Green Tea Rg                11192.50
                 Lemon Grass Rg                       10812.50
                 Spicy Eye Opener Chai Rg             10636.05
                 English Breakfast Rg                 10500.00

[80 rows x 1 columns]

### Different ways to display multi-level index without doing any computation

- just exploring some functions out here. 

In [25]:
df.groupby(['product_category','product_type']).size()

product_category    product_type         
Bakery              Biscotti                  5711
                    Pastry                    6912
                    Scone                    10173
Branded             Clothing                   221
                    Housewares                 526
Coffee              Barista Espresso         16403
                    Drip coffee               8477
                    Gourmet brewed coffee    16912
                    Organic brewed coffee     8489
                    Premium brewed coffee     8135
Coffee beans        Espresso Beans             319
                    Gourmet Beans              366
                    Green beans                134
                    House blend Beans          183
                    Organic Beans              415
                    Premium Beans              336
Drinking Chocolate  Hot chocolate            11468
Flavours            Regular syrup             4979
                    Sugar free syrup    

In [26]:
g = df.groupby(['product_category','product_type'])

list(g.groups.keys())

[('Bakery', 'Biscotti'),
 ('Bakery', 'Pastry'),
 ('Bakery', 'Scone'),
 ('Branded', 'Clothing'),
 ('Branded', 'Housewares'),
 ('Coffee', 'Barista Espresso'),
 ('Coffee', 'Drip coffee'),
 ('Coffee', 'Gourmet brewed coffee'),
 ('Coffee', 'Organic brewed coffee'),
 ('Coffee', 'Premium brewed coffee'),
 ('Coffee beans', 'Espresso Beans'),
 ('Coffee beans', 'Gourmet Beans'),
 ('Coffee beans', 'Green beans'),
 ('Coffee beans', 'House blend Beans'),
 ('Coffee beans', 'Organic Beans'),
 ('Coffee beans', 'Premium Beans'),
 ('Drinking Chocolate', 'Hot chocolate'),
 ('Flavours', 'Regular syrup'),
 ('Flavours', 'Sugar free syrup'),
 ('Loose Tea', 'Black tea'),
 ('Loose Tea', 'Chai tea'),
 ('Loose Tea', 'Green tea'),
 ('Loose Tea', 'Herbal tea'),
 ('Packaged Chocolate', 'Drinking Chocolate'),
 ('Packaged Chocolate', 'Organic Chocolate'),
 ('Tea', 'Brewed Black tea'),
 ('Tea', 'Brewed Chai tea'),
 ('Tea', 'Brewed Green tea'),
 ('Tea', 'Brewed herbal tea')]

In [27]:
df[['product_category','product_type']].drop_duplicates().sort_values(
    ['product_category','product_type']
)

,product_category,product_type
22,Bakery,Biscotti
28,Bakery,Pastry
5,Bakery,Scone
4598,Branded,Clothing
4033,Branded,Housewares
8,Coffee,Barista Espresso
3,Coffee,Drip coffee
0,Coffee,Gourmet brewed coffee
31,Coffee,Organic brewed coffee
57,Coffee,Premium brewed coffee


### Total sales per product type specifically

In [28]:
df.groupby(['product_category','product_type']).agg(Total_Sales= ('Total_Amount','sum')).groupby(level=0 , group_keys=False).apply(
    lambda x : x.sort_values('Total_Sales', ascending=False)
)

Total_Sales
product_category   product_type                      
Bakery             Scone                     36866.12
                   Pastry                    25655.99
                   Biscotti                  19793.53
Branded            Housewares                 7444.00
                   Clothing                   6163.00
Coffee             Barista Espresso          91406.20
                   Gourmet brewed coffee     70034.60
                   Premium brewed coffee     38781.15
                   Organic brewed coffee     37746.50
                   Drip coffee               31984.00
Coffee beans       Premium Beans             14583.50
                   Organic Beans              8509.50
                   Gourmet Beans              6798.00
                   Espresso Beans             5560.25
                   House blend Beans          3294.00
                   Green beans                1340.00
Drinking Chocolate Hot chocolate             72416.00
Flavours           Regular syrup              6084.80
                   Sugar free syrup           2324.00
Loose Tea          Chai tea                   4301.25
                   Herbal tea                 2729.75
                   Black tea                  2711.85
                   Green tea                  1470.75
Packaged Chocolate Drinking Chocolate         2728.04
                   Organic Chocolate          1679.60
Tea                Brewed Chai tea           77081.95
                   Brewed Black tea          47932.00
                   Brewed herbal tea         47539.50
                   Brewed Green tea          23852.50

### Total Sales per month for all stores

In [29]:
df.groupby('Month',observed=True).agg(Sales_Per_Month=('Total_Amount','sum'))

,Sales_Per_Month
Month,
January,81677.74
February,76145.19
March,98834.68
April,118941.08
May,156727.76
June,166485.88


### Total Sales per month for per store

In [30]:
df.pivot_table(index='Month',
              columns ='Store_Name',
              values ='Total_Amount',
              aggfunc ='sum',
              observed = True)

Store_Name,BigBrew,Dunkin,Starbucks
Month,,,
January,27820.65,26543.43,27313.66
February,25719.80,25320.05,25105.34
March,33110.57,32888.68,32835.43
April,40304.14,39159.33,39477.61
May,52598.93,51700.07,52428.76
June,56957.08,54445.69,55083.11


In [31]:
# Take note, we can also use Margin=True to total it automatically. 
# We created a query to specify which month to check only and which store.

Dunk_Jan = df.query("Month == 'January' and Store_Name == 'Dunkin'")

dj = Dunk_Jan.pivot_table(index='Hour',
              columns='Day',
              values='Total_Amount',
              aggfunc='sum',
               observed= True,
                         fill_value=0)
dj

#This is a pivot table for all stores showing their hourly sales each day.

Day,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Hour,,,,,,,
6,222.00,392.30,381.85,400.95,0.00,22.35,250.80
7,447.88,523.23,400.98,376.15,611.00,512.85,456.43
8,581.51,491.85,293.35,356.90,500.20,478.45,449.95
9,647.43,557.60,427.70,428.20,407.35,518.90,597.30
10,568.54,563.65,294.30,411.80,507.10,511.65,560.70
11,206.81,337.85,134.50,268.25,202.90,149.70,274.75
12,279.59,214.65,229.55,213.70,200.70,166.65,262.10
13,127.48,214.10,238.10,182.15,144.50,226.05,306.30
14,252.13,295.40,212.60,198.20,226.85,165.05,245.55


In [32]:
# We validated if these hours on this day really has really no sales because it shows NaN on our Pivot Table.
Dunk_Jan.query("Day=='Friday' and Hour == [6,19,20]")

,transaction_id,transaction_date,transaction_time,transaction_qty,Store_Name,store_location,product_id,unit_price,product_category,product_type,product_detail,Month,Month_Number,Year,Total_Amount,Timestamp_Convert,Hour,Day,Day_parts


In [33]:
# Created a copy for some experimentation. 
dj1 = dj

In [34]:
dj1['Average_Hourly_Sales'] = dj.mean(axis=1).round(2)
#dj1['Average_Hourly_Sales'] = dj.mean(axis=1).map(lambda x: f"{x:,.2f}") This looks cleaner but will have an issue with sorting values

In [35]:
dj1['Hour_Sales'] =  dj.sum(axis=1).round(2)
#dj1['Hour_Sales'] =  dj.sum(axis=1,numeric_only=True).map(lambda x: f"{x:,.2f}") Same thing with Average_Hourly_Sales


In [36]:
dj1.sort_values(by='Average_Hourly_Sales',ascending=False)

Day,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,Average_Hourly_Sales,Hour_Sales
Hour,,,,,,,,,
9,647.43,557.60,427.70,428.20,407.35,518.90,597.30,512.07,4096.55
10,568.54,563.65,294.30,411.80,507.10,511.65,560.70,488.25,3905.99
7,447.88,523.23,400.98,376.15,611.00,512.85,456.43,475.50,3804.02
8,581.51,491.85,293.35,356.90,500.20,478.45,449.95,450.32,3602.53
6,222.00,392.30,381.85,400.95,0.00,22.35,250.80,238.61,1908.86
15,206.79,264.95,244.80,209.55,336.65,124.80,212.30,228.55,1828.39
14,252.13,295.40,212.60,198.20,226.85,165.05,245.55,227.97,1823.75
11,206.81,337.85,134.50,268.25,202.90,149.70,274.75,224.97,1799.73
12,279.59,214.65,229.55,213.70,200.70,166.65,262.10,223.85,1790.79


In [37]:
D_store = df[df.Store_Name.str.contains('Dunkin')]

D_pivot = D_store.pivot_table(index='Hour',
              columns='Day',
              values='Total_Amount',
              aggfunc='sum',
               observed= True,
                margins=True,
                margins_name='Total')

A1 = D_pivot.drop(index='Total').sort_values('Total',ascending=False)

In [38]:
B_store = df[df.Store_Name.str.contains('BigBrew')]

B_pivot = B_store.pivot_table(index='Hour',
              columns='Day',
              values='Total_Amount',
              aggfunc='sum',
               observed= True,
                margins=True,
                margins_name='Total')

A2 = B_pivot.drop(index='Total').sort_values('Total',ascending=False)

In [39]:
S_store = df[df.Store_Name.str.contains('Starbucks')]

S_pivot = S_store.pivot_table(index='Hour',
              columns='Day',
              values='Total_Amount',
              aggfunc='sum',
               observed= True,
                margins=True,
                margins_name='Total')

A3 = S_pivot.drop(index='Total').sort_values('Total',ascending=False)

In [40]:
Agik = pd.concat({
    'Dunkin': pd.Series(A1.index), 
    'BigbBrew': pd.Series(A2.index), 
    'Starbucks': pd.Series(A3.index)
}, axis=1)
Agik.index = Agik.index + 1
print(f"\n{'Ranking of Total Highest sales per hour'}\n")

print(f"{Agik} \n\nWe can clearly see here what hour is peak for each store")


Ranking of Total Highest sales per hour

   Dunkin BigbBrew Starbucks
1      10       10        10
2       9        9         9
3       7        8         8
4       8       11         7
5      15        7        19
6       6       17        16
7      14       14        18
8      12       13        13
9      16       16        17
10     11       18        12
11     13       12        15
12     17       15        11
13     18       19        14
14     19        6       NaN
15     20       20       NaN 

We can clearly see here what hour is peak for each store


In [41]:
#Smart way to create Pivot tables for three or multiple tables at once.

stores = ['Dunkin','Starbucks','BigBrew']

Pibot = {
    store: (
        df[df['Store_Name'].str.contains(store)]
        .pivot_table(
            index='Hour',
            columns='Day',
            values='Total_Amount',
            aggfunc='sum',
            observed=True,
            margins=True,
            margins_name='Total'
        )
    )
    for store in stores
}

for dunk in Pibot.values():
    dunk['Average_Hourly_Sales'] = (
        dunk.drop(columns='Total').mean(axis=1)
    )

In [42]:
Pibot

{'Dunkin': Day      Monday   Tuesday  Wednesday  Thursday    Friday  Saturday    Sunday  \
 Hour                                                                           
 6       2025.95   1465.65    2387.65   2293.55   2108.30   1919.75   2168.25   
 7       4231.11   4433.81    3934.60   3606.49   4149.99   4276.33   3904.29   
 8       4283.86   4324.46    3980.75   4003.23   3919.41   4058.80   3779.02   
 9       4071.18   4509.44    3810.50   3951.47   4462.99   3961.60   4344.49   
 10      4411.62   4673.04    4208.03   4556.90   4519.55   4217.93   4054.39   
 11      2019.86   1674.27    1908.35   1880.00   2005.49   1733.25   1673.25   
 12      2090.99   1701.24    1802.80   2005.13   1853.04   1929.50   1785.79   
 13      1696.63   1678.63    1906.10   1678.43   1591.66   1882.55   1915.48   
 14      2193.81   1977.33    1880.53   1955.55   2036.68   1909.61   1878.46   
 15      2274.14   2062.44    2076.26   2132.68   2101.48   2082.40   2040.50   
 16      2039.18  

In [43]:
Pibot['Dunkin']

Day,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday,Total,Average_Hourly_Sales
Hour,,,,,,,,,
6,2025.95,1465.65,2387.65,2293.55,2108.30,1919.75,2168.25,14369.10,2052.728571
7,4231.11,4433.81,3934.60,3606.49,4149.99,4276.33,3904.29,28536.62,4076.660000
8,4283.86,4324.46,3980.75,4003.23,3919.41,4058.80,3779.02,28349.53,4049.932857
9,4071.18,4509.44,3810.50,3951.47,4462.99,3961.60,4344.49,29111.67,4158.810000
10,4411.62,4673.04,4208.03,4556.90,4519.55,4217.93,4054.39,30641.46,4377.351429
11,2019.86,1674.27,1908.35,1880.00,2005.49,1733.25,1673.25,12894.47,1842.067143
12,2090.99,1701.24,1802.80,2005.13,1853.04,1929.50,1785.79,13168.49,1881.212857
13,1696.63,1678.63,1906.10,1678.43,1591.66,1882.55,1915.48,12349.48,1764.211429
14,2193.81,1977.33,1880.53,1955.55,2036.68,1909.61,1878.46,13831.97,1975.995714


In [44]:
#Created variable that stores sales for Dunkin Shop only
D_only = df[df.Store_Name=='Dunkin']

In [45]:
DD = D_only.pivot_table(
    index='Month',
    columns='Day_parts',
    values='Total_Amount',
    aggfunc='sum',
    observed=True,
    margins=True
)
DD

Day_parts,Early Morning,Mid-Morning,Lunch,Afternoon,Evening,All
Month,,,,,,
January,4998.77,10154.43,4580.38,5980.95,828.90,26543.43
February,4838.93,9438.44,4306.23,5937.04,799.41,25320.05
March,6209.38,12388.06,5557.03,7609.08,1125.13,32888.68
April,7113.46,15311.13,6441.07,9111.46,1182.21,39159.33
May,9775.94,19841.96,8439.99,11941.96,1700.22,51700.07
June,9969.24,20968.64,9087.74,12548.31,1871.76,54445.69
All,42905.72,88102.66,38412.44,53128.80,7507.63,230057.25


In [46]:
# We are just trying to get the Contribution Percentage per Month from the total sales.
D1 = DD.T.drop(index='All',columns='All')
D1['Contribution'] =  (D1.sum(axis=1) / D1.sum(axis=1).sum() * 100).round(2).astype(str) + '%'
D1

Month,January,February,March,April,May,June,Contribution
Day_parts,,,,,,,
Early Morning,4998.77,4838.93,6209.38,7113.46,9775.94,9969.24,18.65%
Mid-Morning,10154.43,9438.44,12388.06,15311.13,19841.96,20968.64,38.3%
Lunch,4580.38,4306.23,5557.03,6441.07,8439.99,9087.74,16.7%
Afternoon,5980.95,5937.04,7609.08,9111.46,11941.96,12548.31,23.09%
Evening,828.90,799.41,1125.13,1182.21,1700.22,1871.76,3.26%


In [47]:
# Take note to use nunique instead of count for specific columns when you want to gather the correct count of unique values only 
# as count will count every row, even if values are just the same as the previous row.

df.groupby('Store_Name').agg(
    Revenue = ('Total_Amount','sum'),
    Order_Count=('transaction_id','nunique'),
    Quantity_Sold=('transaction_qty','sum')).assign(
    AOV = lambda x: x.Revenue / x.Order_Count,
    Avg_Items_Per_Order=lambda x:
        x['Quantity_Sold'] / x['Order_Count'])

# This is a great overview of Revenue, Order Count, Quantity Sold, and AOV. 

,Revenue,Order_Count,Quantity_Sold,AOV,Avg_Items_Per_Order
Store_Name,,,,,
BigBrew,236511.17,50735,71737,4.661696,1.413955
Dunkin,230057.25,47782,71742,4.814726,1.501444
Starbucks,232243.91,50599,70991,4.589891,1.403012


In [48]:
# I did check the price per product because I am wondering why the AOV output is low. It shows that there are a lot of products included 
# that have really low prices, for example, the syrup and some other products as well. 

AOV_Check = df[['product_detail','unit_price']].drop_duplicates().set_index('product_detail').sort_values('unit_price')

# Use this function when you want to return all rows when the output is collapsed due to being too long.
# with pd.option_context('display.max_rows', None):
    #display(AOV_Check)
AOV_Check

,unit_price
product_detail,
Carmel syrup,0.8
Hazelnut syrup,0.8
Chocolate syrup,0.8
Sugar Free Vanilla syrup,0.8
Our Old Time Diner Blend Sm,2.0
...,...
I Need My Bean! Latte cup,23.0
I Need My Bean! Diner mug,23.0
Organic Decaf Blend,28.0


## 6. Export Clean Data
Exporting the cleaned dataset as CSV for use in Power BI dashboard.

In [49]:
df.to_csv('coffee_sales_cleaned.csv', index=False)
print(f"✅ Exported successfully!")
print(f"📊 Final Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📁 Saved as: coffee_sales_cleaned.csv")

✅ Exported successfully!
📊 Final Shape: 149,116 rows, 19 columns
📁 Saved as: coffee_sales_cleaned.csv
